In [52]:
import sys
print(sys.executable)

c:\Users\DELL\AppData\Local\Programs\Python\Python313\python.exe


In [53]:
!{sys.executable} -m pip install imbalanced-learn


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import StandardScaler, LabelEncoder

from imblearn.over_sampling import RandomOverSampler

file_path = "D:\ict AI ML\Student_exam_performance\student_exam_performance.csv"
df = pd.read_csv(file_path)

print(df.shape)
df.head()

<>:10: SyntaxWarning: invalid escape sequence '\i'
<>:10: SyntaxWarning: invalid escape sequence '\i'
C:\Users\DELL\AppData\Local\Temp\ipykernel_23520\3465528227.py:10: SyntaxWarning: invalid escape sequence '\i'
  file_path = "D:\ict AI ML\Student_exam_performance\student_exam_performance.csv"


(100000, 44)


,student_id,age,gender,education_level,school_type,family_income,parent_education,urban_rural,previous_exam_score,previous_gpa,...,exam_difficulty,exam_preparation_days,questions_attempted,questions_correct,time_management_score,exam_anxiety_level,exam_score,performance_grade,pass_status,performance_level
0,STU_000001,20,Female,High School,Public,Middle,NaN,Rural,78.36,2.95,...,Medium,27,95,86,NaN,9.39,90.40,A,Pass,High
1,STU_000002,17,Male,High School,Public,Middle,NaN,Suburban,73.40,2.89,...,Easy,4,100,86,87.17,6.31,86.21,A,Pass,High
2,STU_000003,18,Other,Undergraduate,Public,Middle,Master,Urban,82.76,3.35,...,Easy,28,96,74,63.77,9.87,76.11,B,Pass,Medium
3,STU_000004,20,Male,High School,Public,Middle,High School,Suburban,60.61,2.57,...,Medium,16,97,77,59.51,9.68,79.74,B,Pass,Medium
4,STU_000005,16,Female,High School,Private,High,Bachelor,Suburban,74.79,2.84,...,Medium,28,98,52,84.64,10.00,50.66,D,Pass,Low


In [55]:
leakage_cols = [
    "student_id",
    "exam_score",
    "performance_grade",
    "performance_level",
    "questions_correct",
    "questions_attempted"
]

data = df.drop(columns=leakage_cols)

In [56]:
X = data.drop(columns="pass_status")
y = data["pass_status"]

In [57]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [58]:
numerical_cols = X_train.select_dtypes(exclude="object").columns.tolist()
categorical_cols = X_train.select_dtypes(include="object").columns.tolist()

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

Numerical columns: ['age', 'previous_exam_score', 'previous_gpa', 'attendance_percentage', 'assignment_completion_rate', 'study_hours_per_day', 'self_study_hours', 'private_tuition', 'online_learning_hours', 'practice_tests_completed', 'sleep_hours', 'daily_screen_time', 'physical_activity_hours', 'stress_level', 'internet_access', 'online_course_hours', 'exam_preparation_days', 'time_management_score', 'exam_anxiety_level']
Categorical columns: ['gender', 'education_level', 'school_type', 'family_income', 'parent_education', 'urban_rural', 'class_participation', 'study_consistency', 'study_environment', 'study_method', 'revision_frequency', 'notes_quality', 'sleep_quality', 'break_frequency', 'motivation_level', 'device_availability', 'educational_app_usage', 'exam_difficulty']


In [59]:
ordinal_cols = [
    "education_level",
    "family_income",
    "parent_education",
    "class_participation",
    "study_consistency",
    "revision_frequency",
    "notes_quality",
    "sleep_quality",
    "break_frequency",
    "motivation_level",
    "device_availability",
    "educational_app_usage",
    "exam_difficulty"
]

In [60]:
onehot_cols = [
    col for col in categorical_cols
    if col not in ordinal_cols
]

In [61]:
ordinal_categories = [
    ["High School", "Undergraduate"],

    ["Low", "Lower-Middle", "Middle", "Upper-Middle", "High"],

    ["High School", "Associate", "Bachelor", "Master", "Doctorate"],

    ["Low", "Medium", "High"],

    ["Low", "Medium", "High"],

    ["Rarely", "Weekly", "Daily"],

    ["Poor", "Average", "Excellent"],

    ["Poor", "Fair", "Good", "Excellent"],

    ["Rarely", "Occasionally", "Frequently"],

    ["Low", "Medium", "High"],

    ["Shared", "Dedicated"],

    ["Low", "Moderate", "High"],

    ["Easy", "Medium", "Hard"]
]

In [62]:
num_imputer = SimpleImputer(strategy="median")

X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train[numerical_cols]),
    columns=numerical_cols,
    index=X_train.index
)

X_test_num = pd.DataFrame(
    num_imputer.transform(X_test[numerical_cols]),
    columns=numerical_cols,
    index=X_test.index
)

In [63]:
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_cat = pd.DataFrame(
    cat_imputer.fit_transform(X_train[categorical_cols]),
    columns=categorical_cols,
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    cat_imputer.transform(X_test[categorical_cols]),
    columns=categorical_cols,
    index=X_test.index
)

In [64]:
scaler = StandardScaler()

X_train_num_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_num),
    columns=numerical_cols,
    index=X_train.index
)

X_test_num_scaled = pd.DataFrame(
    scaler.transform(X_test_num),
    columns=numerical_cols,
    index=X_test.index
)

In [65]:
ohe = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_ohe = pd.DataFrame(
    ohe.fit_transform(X_train_cat[onehot_cols]),
    columns=ohe.get_feature_names_out(onehot_cols),
    index=X_train.index
)

X_test_ohe = pd.DataFrame(
    ohe.transform(X_test_cat[onehot_cols]),
    columns=ohe.get_feature_names_out(onehot_cols),
    index=X_test.index
)

In [66]:
ordinal_encoder = OrdinalEncoder(
    categories=ordinal_categories,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train_ord = pd.DataFrame(
    ordinal_encoder.fit_transform(X_train_cat[ordinal_cols]),
    columns=ordinal_cols,
    index=X_train.index
)

X_test_ord = pd.DataFrame(
    ordinal_encoder.transform(X_test_cat[ordinal_cols]),
    columns=ordinal_cols,
    index=X_test.index
)

In [67]:
X_train_cat_encoded = pd.concat(
    [X_train_ohe, X_train_ord],
    axis=1
)

X_test_cat_encoded = pd.concat(
    [X_test_ohe, X_test_ord],
    axis=1
)


In [68]:
X_train_final = pd.concat(
    [X_train_num_scaled, X_train_cat_encoded],
    axis=1
)

X_test_final = pd.concat(
    [X_test_num_scaled, X_test_cat_encoded],
    axis=1
)

print("X_train_final:", X_train_final.shape)
print("X_test_final :", X_test_final.shape)

X_train_final: (80000, 49)
X_test_final : (20000, 49)


In [69]:
label_encoder = LabelEncoder()

y_train_encoded = pd.Series(
    label_encoder.fit_transform(y_train),
    index=y_train.index,
    name="pass_status"
)

y_test_encoded = pd.Series(
    label_encoder.transform(y_test),
    index=y_test.index,
    name="pass_status"
)

print("Classes:", label_encoder.classes_)

Classes: ['Fail' 'Pass']


In [70]:
print(y_train_encoded.value_counts())
print(y_test_encoded.value_counts())

pass_status
1    61911
0    18089
Name: count, dtype: int64
pass_status
1    15478
0     4522
Name: count, dtype: int64


In [71]:
ros = RandomOverSampler(random_state=42)

X_train_balanced, y_train_balanced = ros.fit_resample(
    X_train_final,
    y_train_encoded
)

In [72]:
X_train_balanced = pd.DataFrame(
    X_train_balanced,
    columns=X_train_final.columns
)

y_train_balanced = pd.Series(
    y_train_balanced,
    name="pass_status"
)

In [73]:
print(y_train_balanced.value_counts())

pass_status
1    61911
0    61911
Name: count, dtype: int64
